# Bayesian Model Selection via Marginal Likelihood

Comparing models with different levels of parameter sharing using Bayesian evidence.

## Models Compared

- **Model A**: All dimensions from Bernoulli(0.5) - simplest, no parameters
- **Model B**: All dimensions share one unknown p - intermediate complexity
- **Model C**: Each dimension has separate p_d - most complex

Uses marginal likelihood (evidence) to compute posterior probabilities.

In [1]:
import numpy as np
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.model_selection import model_A, model_B, model_C, compare_models, print_model_comparison

In [2]:
# Load data
X = np.loadtxt('../data/binarydigits.txt')
print(f"Dataset: {X.shape[0]} samples × {X.shape[1]} dimensions\n")

Dataset: 100 samples × 64 dimensions



In [3]:
# Compare all models
log_ml, posteriors = print_model_comparison(X)

Bayesian Model Selection
Data: 100 samples × 64 dimensions

Model Descriptions:
  Model A: All dimensions Bernoulli(0.5)
  Model B: All dimensions share one unknown p
  Model C: Each dimension has separate unknown p_d

Results:
Model    Log Marginal Likelihood   Posterior Probability
----------------------------------------------------------------------
A                        -4436.14              0.0000
B                        -4283.72              0.0000
C                        -3851.20              1.0000

Best Model: C (Each dimension has separate unknown p_d)
Posterior Probability: 1.0000

Bayes Factors (relative to Model A):
  BF(B vs A) = 1.57e+66
  BF(C vs A) = 1.09e+254

Interpretation:
  Strong evidence for Model C (separate parameters)


## Interpretation

The posterior probabilities tell us:
- Which model best explains the observed data
- The Bayesian Occam's Razor effect: more complex models are penalized
- Model C typically wins because binary digits have heterogeneous patterns

### Why Such Extreme Bayes Factors?

The Bayes factors (10^66, 10^254) may seem unrealistically large, but they are mathematically correct:

1. **Evidence accumulates multiplicatively**: With 100 samples × 64 dimensions = 6,400 observations, even small per-observation likelihood differences compound dramatically

2. **Model A is severely misspecified**: Assuming p=0.5 for corner pixels that are almost always 0 (or 1) incurs massive likelihood penalties

3. **Log-scale perspective**: The log Bayes factor of ~585 means each of the 6,400 observations contributes ~0.09 on average - a modest per-observation difference that compounds to astronomical totals

4. **Posterior ≈ 1.0000**: This is numerical precision, not exactly 1.0. The true posterior is something like 0.9999...9 with hundreds of 9s.

**This is expected behavior** when comparing models on moderately-sized datasets where the true data-generating process strongly favors one model.

## References & Summary

### Key Contributions in This Notebook

1. **Bayesian Model Selection**: Comparing three models of different complexity
   - Model A: Fixed p_d = 0.5 (no parameters to learn)
   - Model B: Single shared p for all dimensions (1 parameter)
   - Model C: Separate p_d for each dimension (D parameters)

2. **Marginal Likelihood**: Computing evidence for each model via integration over parameters

3. **Occam's Razor**: Automatic complexity penalty - simpler models preferred unless data strongly supports complexity

### Mathematical Framework

**Model Evidence (Marginal Likelihood)**:
$$p(\mathcal{D}|\mathcal{M}) = \int p(\mathcal{D}|\theta, \mathcal{M}) p(\theta|\mathcal{M}) d\theta$$

**Posterior Model Probability** (with uniform prior over models):
$$p(\mathcal{M}|\mathcal{D}) = \frac{p(\mathcal{D}|\mathcal{M})}{\sum_i p(\mathcal{D}|\mathcal{M}_i)}$$

**Beta-Bernoulli Conjugacy**:
$$p(\mathcal{D}|\mathcal{M}) = \frac{B(\alpha + n_1, \beta + n_0)}{B(\alpha, \beta)}$$

where $B(\cdot, \cdot)$ is the Beta function, $n_1 = \sum x_n$ (count of ones), $n_0 = N - n_1$ (count of zeros).

### References

1. **Bayesian Model Selection**:
   - MacKay, D.J.C. (2003). *Information Theory, Inference, and Learning Algorithms*. Cambridge University Press. (Chapter 28: Model Comparison and Occam's Razor)
   - Kass, R.E. & Raftery, A.E. (1995). "Bayes Factors". *Journal of the American Statistical Association*, 90(430): 773-795.

2. **Occam's Razor in Bayesian Framework**:
   - Jefferys, W.H. & Berger, J.O. (1992). "Ockham's Razor and Bayesian Analysis". *American Scientist*, 80(1): 64-72.
   - Rasmussen, C.E. & Ghahramani, Z. (2001). "Occam's Razor". *Advances in Neural Information Processing Systems 13*: 294-300.

3. **Marginal Likelihood Computation**:
   - Gelman, A. et al. (2013). *Bayesian Data Analysis* (3rd ed.). Chapman & Hall/CRC. (Chapter 7: Evaluating, Comparing, and Expanding Models)

4. **Beta-Bernoulli Model**:
   - Bishop, C.M. (2006). *Pattern Recognition and Machine Learning*. Springer. (Section 2.1: Binary Variables)
   - Murphy, K.P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. (Section 3.3: The Beta-Binomial Model)

5. **Model Complexity**:
   - Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. (Chapter 7: Model Assessment and Selection)

### Summary

This notebook demonstrates **Bayesian Occam's Razor** in action:

- **Model A** (fixed p=0.5): Too restrictive, poor fit to data with varying feature probabilities
- **Model B** (shared p): Trades off simplicity with flexibility, but assumes all dimensions have identical statistics
- **Model C** (separate p_d): Most flexible, but pays complexity penalty - only wins if data truly needs D independent parameters

The **marginal likelihood** automatically balances:
- **Fit quality**: How well the model explains the data
- **Complexity penalty**: Simpler models preferred unless complexity is justified

This is fundamentally different from information criteria (AIC/BIC) which approximate Bayesian model selection but use point estimates rather than full parameter integration. The winning model depends on which best balances fit and complexity for the observed data. If all dimensions truly have similar statistics, Model B would be preferred. Here, the heterogeneous pixel statistics strongly favor Model C despite its complexity penalty.

---

**Next**: [EM Mixture Models](03_em_mixture_models.ipynb) - learning multiple clusters with latent variables.